<div>
<center><img src="../assets/Flux-logo.svg" width="360"/></center>
</div>

<div style="background:linear-gradient(90deg,#036291 0%,#91C2D8 100%);padding:20px 26px;border-radius:10px;border-left:10px solid #D9A441;margin-top:18px">
<h1 style="margin:0;color:#ffffff">Module 2: Kubeflow Trainer</h1>
<p style="margin:6px 0 0 0;color:#DCECF4;font-size:15px">AI/ML and HPC simulation from one control plane</p>
<p style="margin:2px 0 0 0;color:#DCECF4;font-size:13px">SC26 &middot; Chicago &middot; November 2026</p>
</div>

Same Usernetes cluster, third thing installed into it. The
[Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/getting-started/) runs
AI/ML workloads in Kubernetes directly from Python. We start with a plain PyTorch training
job, then swap in the Flux runtime so the same `TrainJob` abstraction launches an MPI
simulation instead.


## 1. Install the Kubeflow Trainer

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> Running AI/ML training jobs in Kubernetes is a first class citizen.
</div>

The newly released [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/getting-started/) project makes it easy to run Kubernetes components, specifically for Artificial Intelligence and Machine Learning workloads (AI/ML) in Kubernetes directly from Python. In your terminal to the left, use `kubectl` to install the Kubeflow Training Operator: 

```bash
# This is for JobSet and the Trainer Manager
VERSION=v2.0.0
kubectl apply --server-side -k "https://github.com/kubeflow/trainer.git/manifests/overlays/manager?ref=${VERSION}"

# This is for the runtimes (we need to give the webhook some time to create)
sleep 5
kubectl apply --server-side -k "https://github.com/kubeflow/trainer.git/manifests/overlays/runtimes?ref=${VERSION}"
```

If you get a `failed calling webhook` error, wait 30 seconds and try again. Hooks sometimes have a delay in starting up.
Next, let's run Mnist training. Mnist is a well-known [dataset of handwritten digits](https://en.wikipedia.org/wiki/MNIST_database) used for machine learning tasks. We will be using Kubeflow, and using YAML for this tutorial. Note that if you like Python, the entire interaction of using Kubeflow in Kubernetes can be done using the [Python SDK](https://www.kubeflow.org/docs/components/trainer/getting-started/).

## 2. Run MNIST

While we can use the Python SDK for all our interactions, let's create the job the "old school" way - by applying a YAML file. Take a look at [pytorch-mnist.yaml](manifests/pytorch-mnist.yaml) and then in your terminal, run the job in your cluster by using `kubectl apply` with `-f` for a file.

```bash
kubectl apply -f ./manifests/pytorch-mnist.yaml
```

To see the pods, you can use `kubectl get` on the Pod resource type. Note that we will have two. Index 0 is the master, and 1 is the worker.

```bash
kubectl get pods

# More information about the hosts in "output wide" mode
kubectl get pods -o wide
```

And then get the pod identifier and look at the output. The `-f` will keep the output streaming.

```bash
kubectl logs pytorch-simple-node-0-0-xxxx -f
```

When the logs appear to be done, check the pods to see they are `Completed`

```bash
kubectl get pods
```

When you are ready to clean up, just `kubectl delete` the same file. Note that a pod in `Completed` state does not consume resources.

```bash
kubectl delete -f ./manifests/pytorch-mnist.yaml
```

Congratulations - you just ran your first AI/ML Job in User-space Kubernetes!

## 3. Now the same abstraction, but for HPC

<div class="alert alert-block" style="background-color:#91C2D8;color:#06293D">
<span style="font-weight:600">Description:</span> A <code>TrainJob</code> that runs LAMMPS instead of PyTorch, with Flux bootstrapping the ranks.
</div>

This LAMMPS example assumes two small nodes. Retrieve and alter the manifest to increase
the problem size if you have more.

```bash
kubectl apply --server-side -f https://raw.githubusercontent.com/kubeflow/trainer/refs/heads/master/examples/flux/flux-runtime.yaml
kubectl apply -f https://raw.githubusercontent.com/kubeflow/trainer/refs/heads/master/examples/flux/lammps-train-job.yaml
```

<!-- TODO(sc26): this section came from 2026/HPSF, where it ran on a two-node EKS cluster.
     Confirm it works on the single-node ARM usernetes setup, and confirm the images have
     ARM builds. If not, either pin an ARM manifest or cut the section. -->


## 4. Monitor and read the logs

```bash
kubectl get pods -w
```

The lead broker is pod index `0-0`. Watch it bootstrap and then run LAMMPS:

```bash
kubectl logs lammps-flux-node-0-0-<suffix> -c node -f
```

Same `TrainJob` resource, same control plane. Only the runtime changed.


<div style="background:#DCECF4;border-left:6px solid #D9A441;padding:12px 18px;color:#06293D"><strong>Module 2, Notebook 3 complete</strong></div>

Next: [Quantum simulation under Flux](04_quantum_local.ipynb).
